## B4 – LightGBM

Gradient boosting con alberi di regressione (LightGBM), loss L1.
Stessi dati di B2: addestramento su `train_enc`, early stopping e scelta degli iperparametri su `tune_enc`, valutazione finale su `val_enc`.

### 1. Import e caricamento dei dati codificati
Carichiamo i tre split prodotti da `prepare.py` e il file `meta.json` con le informazioni sulle colonne.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

DATA = Path("../data/encoded")

train = pd.read_parquet(DATA / "train_enc.parquet")
tune  = pd.read_parquet(DATA / "tune_enc.parquet")
val   = pd.read_parquet(DATA / "val_enc.parquet")

with open(DATA / "meta.json") as f:
    meta = json.load(f)

print("LightGBM", lgb.__version__)
for nome, df in [("train", train), ("tune", tune), ("val", val)]:
    print(f"{nome:5s} {df.shape}")
print(train.dtypes)
print(list(meta.keys()))

LightGBM 4.7.0
train (3000316, 22)
tune  (1000351, 22)
val   (10069254, 22)
riga                               int64
flow_id                              str
t_start              datetime64[us, UTC]
delta_rtt                        float64
start_rtt                        float64
id_ip_version                      int32
id_experiment                      int32
id_exp_act                         int32
id_application                     int32
id_hostname                        int32
id_server_site                     int32
id_client_site                     int32
id_server_L2                       int32
id_client_L2                       int32
id_porta_servizio                  int32
id_porta_client                    int32
id_verso                           int32
id_verso_incerto                   int32
server_site                          str
client_site                          str
verso_incerto                       bool
an_durata_s                      float64
dtype: object
['sorgen

### 2. Input, target e costruzione dei Dataset LightGBM
- **Input categorici:** le 13 colonne `id_*` (codici interi: 0 PAD, 1 MISSING, 2 RARE, 3 UNK, da 4 in su i valori frequenti), dichiarate a LightGBM come categoriche, quindi niente one-hot.
- **Input numerico:** `start_rtt`, senza trasformazioni (agli alberi interessa solo l'ordine dei valori).
- **Target:** `delta_rtt`.

Il Dataset di tune è costruito con `reference=dtrain`, così usa le stesse soglie di discretizzazione del train.

In [6]:
CAT = sorted(c for c in train.columns if c.startswith("id_"))
NUM = ["start_rtt"]
FEAT = CAT + NUM
TARGET = "delta_rtt"

assert len(CAT) == 13, CAT
assert TARGET in train.columns

def Xy(df):
    return df[FEAT], df[TARGET].to_numpy()

X_tr, y_tr = Xy(train)
X_tu, y_tu = Xy(tune)
X_va, y_va = Xy(val)

dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=CAT,
                     params=dict(feature_pre_filter=False), free_raw_data=False)
dtune  = lgb.Dataset(X_tu, label=y_tu, categorical_feature=CAT, reference=dtrain,
                     params=dict(feature_pre_filter=False), free_raw_data=False)

# quante categorie ha ogni colonna (codice massimo + 1)
print(pd.Series({c: int(train[c].max()) + 1 for c in CAT}).sort_values())

id_ip_version          6
id_verso               6
id_verso_incerto       6
id_porta_client        8
id_experiment         10
id_exp_act            33
id_porta_servizio     50
id_application        60
id_server_site        85
id_client_site        87
id_server_L2         342
id_client_L2         344
id_hostname          690
dtype: int64


### 3. Primo modello (parametri ragionevoli) con early stopping sul tune
Loss L1 (`objective="l1"`): le foglie predicono mediane dei residui.
Si aggiungono alberi finché la MAE sul tune non smette di migliorare per 100 alberi consecutivi; si tiene il numero di alberi migliore.
Serve come punto di partenza per la ricerca degli iperparametri.

In [3]:
import time

params = dict(
    objective="l1",
    metric="l1",
    learning_rate=0.1,
    num_leaves=63,
    min_data_in_leaf=200,
    cat_smooth=10,
    seed=0,
    verbose=-1,
)

t0 = time.perf_counter()
model = lgb.train(
    params,
    dtrain,
    num_boost_round=5000,
    valid_sets=[dtune],
    valid_names=["tune"],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)],
)
print(f"tempo: {time.perf_counter() - t0:.0f} s")
print("alberi migliori:", model.best_iteration)
print("MAE tune:", model.best_score["tune"]["l1"])

Training until validation scores don't improve for 100 rounds
[100]	tune's l1: 1.62469
[200]	tune's l1: 1.60671
[300]	tune's l1: 1.60528
Early stopping, best iteration is:
[254]	tune's l1: 1.60089
tempo: 23 s
alberi migliori: 254
MAE tune: 1.6008869073901706


### 4. Riferimento: B0 sul tune
Per dare una scala alla MAE di LightGBM sul tune, calcoliamo la MAE della baseline B0 (mediana globale del train) sullo stesso insieme.

In [4]:
b0 = np.median(y_tr)
mae_b0_tune = np.mean(np.abs(y_tu - b0))
print(f"B0 mediana train = {b0:.3f}  |  MAE tune B0 = {mae_b0_tune:.3f}  |  LGBM = {model.best_score['tune']['l1']:.3f}")

B0 mediana train = 4.405  |  MAE tune B0 = 3.209  |  LGBM = 1.601


### 5. Ricerca iperparametri (1): complessità degli alberi
Griglia su `num_leaves` (foglie per albero) e `min_data_in_leaf` (flow minimi per foglia), con learning rate fisso a 0,1 e early stopping sul tune.
Differenze di MAE sotto ~0,005 sono considerate rumore; a parità si preferisce il modello più semplice.

In [5]:
from itertools import product

griglia = dict(num_leaves=[31, 63, 127, 255],
               min_data_in_leaf=[100, 1000, 5000])

risultati = []
for nl, md in product(griglia["num_leaves"], griglia["min_data_in_leaf"]):
    p = {**params, "num_leaves": nl, "min_data_in_leaf": md}
    t0 = time.perf_counter()
    m = lgb.train(p, dtrain, num_boost_round=5000,
                  valid_sets=[dtune], valid_names=["tune"],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
    risultati.append(dict(num_leaves=nl, min_data_in_leaf=md,
                          alberi=m.best_iteration,
                          mae_tune=m.best_score["tune"]["l1"],
                          secondi=round(time.perf_counter() - t0)))
    print(risultati[-1])

ris = pd.DataFrame(risultati).sort_values("mae_tune")
ris

[LightGBM] [Fatal] Reducing `min_data_in_leaf` with `feature_pre_filter=true` may cause unexpected behaviour for features that were pre-filtered by the larger `min_data_in_leaf`.
You need to set `feature_pre_filter=false` to dynamically change the `min_data_in_leaf`.
[LightGBM] [Fatal] Reducing `min_data_in_leaf` with `feature_pre_filter=true` may cause unexpected behaviour for features that were pre-filtered by the larger `min_data_in_leaf`.
You need to set `feature_pre_filter=false` to dynamically change the `min_data_in_leaf`.


{'num_leaves': 31, 'min_data_in_leaf': 100, 'alberi': 391, 'mae_tune': np.float64(1.6041163321470826), 'secondi': 26}
{'num_leaves': 31, 'min_data_in_leaf': 1000, 'alberi': 257, 'mae_tune': np.float64(1.6129120926480853), 'secondi': 20}
{'num_leaves': 31, 'min_data_in_leaf': 5000, 'alberi': 943, 'mae_tune': np.float64(1.6514408746111366), 'secondi': 60}
{'num_leaves': 63, 'min_data_in_leaf': 100, 'alberi': 797, 'mae_tune': np.float64(1.600624218711868), 'secondi': 55}
{'num_leaves': 63, 'min_data_in_leaf': 1000, 'alberi': 193, 'mae_tune': np.float64(1.6232029964487582), 'secondi': 21}
{'num_leaves': 63, 'min_data_in_leaf': 5000, 'alberi': 440, 'mae_tune': np.float64(1.6572716123138649), 'secondi': 45}
{'num_leaves': 127, 'min_data_in_leaf': 100, 'alberi': 222, 'mae_tune': np.float64(1.5995749174176357), 'secondi': 26}
{'num_leaves': 127, 'min_data_in_leaf': 1000, 'alberi': 251, 'mae_tune': np.float64(1.6181804420592067), 'secondi': 33}
{'num_leaves': 127, 'min_data_in_leaf': 5000, 'alb

,num_leaves,min_data_in_leaf,alberi,mae_tune,secondi
6,127,100,222,1.599575,26
9,255,100,347,1.599751,52
3,63,100,797,1.600624,55
0,31,100,391,1.604116,26
1,31,1000,257,1.612912,20
7,127,1000,251,1.618180,33
4,63,1000,193,1.623203,21
10,255,1000,376,1.636906,64
8,127,5000,3451,1.642931,450
2,31,5000,943,1.651441,60


### 6. Ricerca iperparametri (2): foglie più piccole
Nella griglia precedente il migliore `min_data_in_leaf` (100) era al bordo, quindi proviamo valori più piccoli (20, 50).
Insieme a `min_data_in_leaf` facciamo scendere anche `min_data_per_group` (minimo di flow per gruppo di categorie in un taglio categorico, default 100), altrimenti i tagli sulle colonne `id_*` resterebbero vincolati come prima.
La configurazione con 100 riproduce quella già vista (controllo).
Definiamo la funzione `prova`, che addestra con early stopping sul tune e restituisce una riga di risultati.

In [7]:
def prova(**modifiche):
    p = {**params, **modifiche}
    t0 = time.perf_counter()
    m = lgb.train(p, dtrain, num_boost_round=5000,
                  valid_sets=[dtune], valid_names=["tune"],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
    riga = dict(**modifiche,
                alberi=m.best_iteration,
                mae_tune=round(m.best_score["tune"]["l1"], 4),
                secondi=round(time.perf_counter() - t0))
    print(riga)
    return riga

risultati2 = []
for nl, md in product([63, 127], [20, 50, 100]):
    risultati2.append(prova(num_leaves=nl, min_data_in_leaf=md, min_data_per_group=md))

ris2 = pd.DataFrame(risultati2).sort_values("mae_tune")
ris2

{'num_leaves': 63, 'min_data_in_leaf': 20, 'min_data_per_group': 20, 'alberi': 295, 'mae_tune': np.float64(1.6075), 'secondi': 25}
{'num_leaves': 63, 'min_data_in_leaf': 50, 'min_data_per_group': 50, 'alberi': 411, 'mae_tune': np.float64(1.5944), 'secondi': 31}
{'num_leaves': 63, 'min_data_in_leaf': 100, 'min_data_per_group': 100, 'alberi': 797, 'mae_tune': np.float64(1.6006), 'secondi': 54}
{'num_leaves': 127, 'min_data_in_leaf': 20, 'min_data_per_group': 20, 'alberi': 264, 'mae_tune': np.float64(1.5952), 'secondi': 31}
{'num_leaves': 127, 'min_data_in_leaf': 50, 'min_data_per_group': 50, 'alberi': 376, 'mae_tune': np.float64(1.6011), 'secondi': 36}
{'num_leaves': 127, 'min_data_in_leaf': 100, 'min_data_per_group': 100, 'alberi': 222, 'mae_tune': np.float64(1.5996), 'secondi': 26}


,num_leaves,min_data_in_leaf,min_data_per_group,alberi,mae_tune,secondi
1,63,50,50,411,1.5944,31
3,127,20,20,264,1.5952,31
5,127,100,100,222,1.5996,26
2,63,100,100,797,1.6006,54
4,127,50,50,376,1.6011,36
0,63,20,20,295,1.6075,25


### 7. Ricerca iperparametri (3): tagli sulle categorie
Fissati `num_leaves=63`, `min_data_in_leaf=50`, `min_data_per_group=50` (le differenze nella griglia precedente sono di circa 0,01, simili al rumore: siamo su un plateau).
Proviamo `cat_smooth` (regolarizzazione delle categorie poco frequenti) e `max_cat_threshold` (numero massimo di categorie da una parte di un taglio).
Si cambia configurazione solo per un miglioramento > 0,01.

In [8]:
base = dict(num_leaves=63, min_data_in_leaf=50, min_data_per_group=50)

risultati3 = []
for cs, mct in product([1, 10, 50], [32, 64]):
    risultati3.append(prova(**base, cat_smooth=cs, max_cat_threshold=mct))

ris3 = pd.DataFrame(risultati3).sort_values("mae_tune")
ris3

{'num_leaves': 63, 'min_data_in_leaf': 50, 'min_data_per_group': 50, 'cat_smooth': 1, 'max_cat_threshold': 32, 'alberi': 352, 'mae_tune': np.float64(1.6074), 'secondi': 28}
{'num_leaves': 63, 'min_data_in_leaf': 50, 'min_data_per_group': 50, 'cat_smooth': 1, 'max_cat_threshold': 64, 'alberi': 658, 'mae_tune': np.float64(1.5857), 'secondi': 44}
{'num_leaves': 63, 'min_data_in_leaf': 50, 'min_data_per_group': 50, 'cat_smooth': 10, 'max_cat_threshold': 32, 'alberi': 411, 'mae_tune': np.float64(1.5944), 'secondi': 32}
{'num_leaves': 63, 'min_data_in_leaf': 50, 'min_data_per_group': 50, 'cat_smooth': 10, 'max_cat_threshold': 64, 'alberi': 293, 'mae_tune': np.float64(1.5959), 'secondi': 26}
{'num_leaves': 63, 'min_data_in_leaf': 50, 'min_data_per_group': 50, 'cat_smooth': 50, 'max_cat_threshold': 32, 'alberi': 573, 'mae_tune': np.float64(1.6044), 'secondi': 42}
{'num_leaves': 63, 'min_data_in_leaf': 50, 'min_data_per_group': 50, 'cat_smooth': 50, 'max_cat_threshold': 64, 'alberi': 313, 'mae_

,num_leaves,min_data_in_leaf,min_data_per_group,cat_smooth,max_cat_threshold,alberi,mae_tune,secondi
1,63,50,50,1,64,658,1.5857,44
2,63,50,50,10,32,411,1.5944,32
5,63,50,50,50,64,313,1.5945,29
3,63,50,50,10,64,293,1.5959,26
4,63,50,50,50,32,573,1.6044,42
0,63,50,50,1,32,352,1.6074,28


### 8. Modello con learning rate più basso
Configurazione scelta: `num_leaves=63`, `min_data_in_leaf=50`, `min_data_per_group=50`, `cat_smooth=10`, `max_cat_threshold=32` (default).
La prova con `cat_smooth=1, max_cat_threshold=64` (1,5857) migliora di meno di 0,01 e in modo incoerente (`cat_smooth=1` con 32 è la peggiore), quindi è considerata rumore.
Abbassiamo il learning rate a 0,03: ogni albero corregge meno, ne servono di più, ma la somma è più stabile. La pazienza dell'early stopping sale a 300 alberi in proporzione.
Il numero di alberi trovato qui servirà per il riaddestramento finale su train+tune.

In [9]:
params_fin = {**params, **base, "learning_rate": 0.03}

t0 = time.perf_counter()
model_lr = lgb.train(params_fin, dtrain, num_boost_round=10000,
                     valid_sets=[dtune], valid_names=["tune"],
                     callbacks=[lgb.early_stopping(300), lgb.log_evaluation(250)])
print(f"tempo: {time.perf_counter() - t0:.0f} s")
print("alberi migliori:", model_lr.best_iteration)
print("MAE tune:", round(model_lr.best_score["tune"]["l1"], 4))

Training until validation scores don't improve for 300 rounds
[250]	tune's l1: 1.64393
[500]	tune's l1: 1.6205
[750]	tune's l1: 1.61097
[1000]	tune's l1: 1.60874
[1250]	tune's l1: 1.60767
[1500]	tune's l1: 1.6071
Early stopping, best iteration is:
[1234]	tune's l1: 1.60539
tempo: 91 s
alberi migliori: 1234
MAE tune: 1.6054


### 9. Modello finale B4: riaddestramento su train + tune
Il learning rate 0,03 (1,6054 sul tune, 1.234 alberi) non migliora quello a 0,1 (1,5944, 411 alberi): la differenza è entro il rumore e il valore 1,5944 è ottimista perché è il migliore di ~20 configurazioni. Teniamo η = 0,1.

Configurazione finale: `learning_rate=0.1`, `num_leaves=63`, `min_data_in_leaf=50`, `min_data_per_group=50`, `cat_smooth=10`, `max_cat_threshold=32`, loss L1, 411 alberi.
Come per B2, riaddestriamo su train + tune (≈ 4 M flow) con il numero di alberi fissato, senza early stopping.

In [10]:
params_b4 = {**params, **base}          # learning_rate resta 0.1
N_ALBERI = 411

X_all = pd.concat([X_tr, X_tu], ignore_index=True)
y_all = np.concatenate([y_tr, y_tu])

dall = lgb.Dataset(X_all, label=y_all, categorical_feature=CAT,
                   params=dict(feature_pre_filter=False))

t0 = time.perf_counter()
b4 = lgb.train(params_b4, dall, num_boost_round=N_ALBERI)
print(f"tempo: {time.perf_counter() - t0:.0f} s  |  alberi: {b4.num_trees()}")

b4.save_model("b4_lightgbm.txt")

tempo: 31 s  |  alberi: 411


### 10. Valutazione di B4 sulla validation
Predizioni del modello finale (addestrato su train + tune) sui 10.069.254 flow della validation, mai usata nelle scelte.
Metriche: MAE (principale), RMSE, P90 dell'errore assoluto (90% dei flow ha errore sotto questo valore).
Riferimenti sulla validation (MAE): B0 3,298 · B1 2,209 · B2 1,966.

In [11]:
def metriche(y, pred):
    err = np.abs(y - pred)
    return dict(n=len(y),
                MAE=round(err.mean(), 3),
                RMSE=round(np.sqrt(np.mean((y - pred) ** 2)), 3),
                P90=round(np.quantile(err, 0.9), 3))

t0 = time.perf_counter()
pred_va = b4.predict(X_va)
print(f"predizione: {time.perf_counter() - t0:.0f} s")

print("B4 validation:", metriche(y_va, pred_va))

# colonne disponibili per i sottogruppi (non usate come input)
print([c for c in val.columns if c not in FEAT + [TARGET]])

predizione: 33 s
B4 validation: {'n': 10069254, 'MAE': np.float64(1.833), 'RMSE': np.float64(47.076), 'P90': np.float64(4.563)}
['riga', 'flow_id', 't_start', 'server_site', 'client_site', 'verso_incerto', 'an_durata_s']


### 11. B4 sui sottogruppi (stessa valutazione delle baseline)
Funzione di valutazione e soglie dell'RMSE robusto copiate identiche dal notebook di B2, così le metriche sono confrontabili.
Le predizioni di B4 sulla validation vengono messe in una tabella con target e colonne dei sottogruppi e registrate in DuckDB come `pred_b4`.

In [51]:
from pathlib import Path
import duckdb

PROJECT_ROOT = Path.cwd().parent                     # il notebook sta in results/
ENC_DIR = PROJECT_ROOT / "data" / "encoded"

con = duckdb.connect()
con.execute("SET threads TO 8")
con.execute("SET memory_limit = '8GB'")

for nome in ["train_enc", "tune_enc", "val_enc"]:
    file = (ENC_DIR / f"{nome}.parquet").as_posix()
    con.execute(f"CREATE OR REPLACE VIEW {nome} AS SELECT * FROM read_parquet('{file}')")

display(con.execute("""
    SELECT 'train_enc' AS insieme, COUNT(*) AS n FROM train_enc
    UNION ALL SELECT 'tune_enc', COUNT(*) FROM tune_enc
    UNION ALL SELECT 'val_enc',  COUNT(*) FROM val_enc
""").df())

display(con.execute("DESCRIBE val_enc").df()[["column_name", "column_type"]])

,insieme,n
0,train_enc,3000316
1,tune_enc,1000351
2,val_enc,10069254


,column_name,column_type
0,riga,BIGINT
1,flow_id,VARCHAR
2,t_start,TIMESTAMP WITH TIME ZONE
3,delta_rtt,DOUBLE
4,start_rtt,DOUBLE
5,id_ip_version,INTEGER
6,id_experiment,INTEGER
7,id_exp_act,INTEGER
8,id_application,INTEGER
9,id_hostname,INTEGER


In [52]:
import json
import numpy as np

meta = json.loads((ENC_DIR / "meta.json").read_text(encoding="utf-8"))
COL_ID = meta["colonne_id"]
print(len(COL_ID), "input categorici:", COL_ID)

def carica(nome):
    df = con.execute(f"SELECT riga, delta_rtt, start_rtt, {', '.join(COL_ID)} FROM {nome}").df()
    riga = df["riga"].to_numpy()
    y    = df["delta_rtt"].to_numpy(np.float32)
    rtt  = df["start_rtt"].to_numpy(np.float32)
    X    = df[COL_ID].to_numpy(np.int32)
    return riga, y, rtt, X

_,       y_tr, rtt_tr, X_tr = carica("train_enc")
_,       y_tu, rtt_tu, X_tu = carica("tune_enc")
riga_va, y_va, rtt_va, X_va = carica("val_enc")

print("forme di X:", X_tr.shape, X_tu.shape, X_va.shape)
print(f"memoria X_va: {X_va.nbytes / 1e9:.2f} GB")
print("mediana di y_tr:", np.median(y_tr))

13 input categorici: ['id_ip_version', 'id_experiment', 'id_exp_act', 'id_application', 'id_hostname', 'id_server_site', 'id_client_site', 'id_server_L2', 'id_client_L2', 'id_porta_servizio', 'id_porta_client', 'id_verso', 'id_verso_incerto']
forme di X: (3000316, 13) (1000351, 13) (10069254, 13)
memoria X_va: 0.52 GB
mediana di y_tr: 4.405


In [53]:
pred_df = pd.DataFrame({"riga": riga_va, "pred": pred_va})
con.register("pred_df", pred_df)

con.execute("""
    CREATE OR REPLACE VIEW pred_b3 AS
    SELECT v.riga, v.delta_rtt, p.pred,
           v.server_site, v.client_site, v.verso_incerto, v.an_durata_s
    FROM val_enc AS v
    LEFT JOIN pred_df AS p ON v.riga = p.riga
""")

controllo = con.execute("""
    SELECT COUNT(*)                            AS flow,
           COUNT(pred)                         AS con_previsione,
           COUNT(*) - COUNT(DISTINCT riga)     AS righe_duplicate
    FROM pred_b3
""").df()
display(controllo)

display(con.execute("SELECT * FROM pred_b3 LIMIT 5").df())

,flow,con_previsione,righe_duplicate
0,10069254,10069254,0


,riga,delta_rtt,pred,server_site,client_site,verso_incerto,an_durata_s
0,737281,0.168,5.158066,CERN-PROD,INFN-T1,False,321.440918
1,737282,5.138,5.062239,CERN-PROD,INFN-T1,False,357.010501
2,737283,18.464,5.486126,CERN-PROD,RAL-LCG2,False,343.313558
3,737284,5.023,5.139163,CERN-PROD,INFN-T1,False,386.950179
4,737285,0.799,4.974524,CERN-PROD,INFN-T1,False,504.856493


In [60]:

# --- Soglie dell'RMSE robusto (train completo, come nel notebook delle baseline) ---
PREPARED = PROJECT_ROOT / "data" / "student_dataset" / "flows_prepared.parquet"
Q_LO, Q_HI = 0.0005, 0.9995
LO, HI = con.execute(f"""
    SELECT quantile_cont(delta_rtt, [{Q_LO}, {Q_HI}])
    FROM read_parquet('{PREPARED.as_posix()}')
    WHERE split = 'train'
""").fetchone()[0]
print(f"LO = {LO:.2f} | HI = {HI:.2f}   (attesi: -184.23 | 58.82)")

NOTA_ROBUSTO = (f"† RMSE robusto: solo flow con delta_rtt in [{LO:.2f}, {HI:.2f}] ms, "
                f"quantili {Q_LO:.2%}–{Q_HI:.2%} del train (insieme fisso, uguale per tutti i modelli).")

# --- Funzione di valutazione (identica al notebook delle baseline) ---
SOTTOGRUPPI = {
    "globale":       "TRUE",
    "FNAL":          "server_site = 'USCMS-FNAL-WC1' OR client_site = 'USCMS-FNAL-WC1'",
    "sito mancante": "server_site = 'MISSING' OR client_site = 'MISSING'",
    "verso incerto": "verso_incerto",
    "flow > 1 min":  "an_durata_s > 60",
}

def valuta(vista):
    """vista: tabella/vista DuckDB con delta_rtt, pred e le colonne dei sottogruppi (solo validation)."""
    parti = []
    for ordine, (nome, cond) in enumerate(SOTTOGRUPPI.items()):
        parti.append(f"""
            SELECT {ordine} AS ordine, '{nome}' AS gruppo, COUNT(*) AS n,
                   AVG(ABS(delta_rtt - pred))                            AS mae,
                   SQRT(AVG(POWER(delta_rtt - pred, 2)))                 AS rmse,
                   SQRT(AVG(POWER(delta_rtt - pred, 2))
                        FILTER (WHERE delta_rtt BETWEEN {LO} AND {HI})) AS rmse_robusto,
                   quantile_cont(ABS(delta_rtt - pred), 0.90)            AS p90_abs,
                   quantile_cont(ABS(delta_rtt - pred), 0.99)            AS p99_abs
            FROM {vista}
            WHERE {cond}
        """)
    query = "SELECT * FROM (" + " UNION ALL ".join(parti) + ") ORDER BY ordine"
    return con.execute(query).df().drop(columns="ordine")

NOMI_METRICHE = {"mae": "MAE", "rmse": "RMSE", "rmse_robusto": "RMSE robusto †",
                 "p90_abs": "P90 |err|", "p99_abs": "P99 |err|"}

# --- Valutazione di B3 ---
ris_b4 = valuta("pred_b3")
display(ris_b4.rename(columns=NOMI_METRICHE).round(3))
print(NOTA_ROBUSTO)

mae_b2 = ris_b4.loc[ris_b4["gruppo"] == "globale", "mae"].iloc[0]
print(f"\nMAE sulla validation → B0: 3.298 | B1: 2.209 | B2: {mae_b2:.3f}")



LO = -184.23 | HI = 58.82   (attesi: -184.23 | 58.82)


,gruppo,n,MAE,RMSE,RMSE robusto †,P90 |err|,P99 |err|
0,globale,10069254,1.833,47.076,4.945,4.563,22.640
1,FNAL,215471,0.996,2.604,2.073,2.499,6.150
2,sito mancante,2122477,1.925,15.310,5.883,4.695,24.515
3,verso incerto,32662,0.262,6.047,1.743,0.172,2.427
4,flow > 1 min,6632723,2.380,57.840,5.747,4.955,26.605


† RMSE robusto: solo flow con delta_rtt in [-184.23, 58.82] ms, quantili 0.05%–99.95% del train (insieme fisso, uguale per tutti i modelli).

MAE sulla validation → B0: 3.298 | B1: 2.209 | B2: 1.833


### 12. Feature importance
Due misure, sommate su tutti i 411 alberi:
- **gain**: riduzione totale della loss ottenuta dai tagli su quella colonna (quanto è utile);
- **split**: numero di volte che la colonna è usata in un taglio (quanto è usata).

Cautele: il gain tende a favorire le colonne con molte categorie (più modi di tagliare); colonne correlate (es. sito e prefisso del server, hostname) si dividono il merito, quindi un'importanza bassa non vuol dire che l'informazione sia inutile, ma che è già coperta da altre colonne.

In [61]:
imp = pd.DataFrame({
    "colonna": b4.feature_name(),
    "gain":    b4.feature_importance(importance_type="gain"),
    "split":   b4.feature_importance(importance_type="split"),
})
imp["gain_%"]  = (100 * imp["gain"]  / imp["gain"].sum()).round(1)
imp["split_%"] = (100 * imp["split"] / imp["split"].sum()).round(1)

imp.sort_values("gain", ascending=False).reset_index(drop=True)

,colonna,gain,split,gain_%,split_%
0,id_application,3.435630e+07,1823,70.1,7.2
1,id_client_L2,6.097270e+06,5420,12.4,21.3
2,id_exp_act,2.914849e+06,1184,5.9,4.6
3,start_rtt,1.722470e+06,3845,3.5,15.1
4,id_server_L2,1.443179e+06,4025,2.9,15.8
5,id_hostname,1.184714e+06,7302,2.4,28.7
6,id_client_site,5.626374e+05,463,1.1,1.8
7,id_porta_servizio,3.384329e+05,926,0.7,3.6
8,id_server_site,3.327441e+05,267,0.7,1.0
9,id_verso,4.473978e+04,141,0.1,0.6


### 13. Salvataggio
Stesso formato di B2, in `results/metriche/`: metriche per sottogruppo (CSV), riepilogo con riga globale e impostazioni (JSON), più la feature importance. Il modello va in `results/modelli/`.
"parametri" per B4 = numero totale di foglie (valori stimati dai dati); la struttura dei tagli non è contata, quindi il confronto con il numero di pesi di B2 è solo indicativo.

In [64]:
OUT_DIR = PROJECT_ROOT / "results" / "metriche"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# metriche per sottogruppo e importanza
ris_b4.to_csv(OUT_DIR / "B4_lightgbm_sottogruppi.csv", index=False)
imp.sort_values("gain", ascending=False).to_csv(OUT_DIR / "B4_lightgbm_importanza.csv", index=False)

TEMPO_TRAIN_B4 = 31   # <-- scrivi i secondi stampati dalla cella 9

# riepilogo: riga globale + impostazioni
glob = ris_b4.set_index("gruppo").loc["globale"]
n_foglie = sum(t["num_leaves"] for t in b4.dump_model()["tree_info"])
riepilogo_b4 = {
    "modello":           "B4 LightGBM (L1)",
    "MAE":               float(glob["mae"]),
    "RMSE":              float(glob["rmse"]),
    "RMSE_robusto":      float(glob["rmse_robusto"]),
    "P90_abs":           float(glob["p90_abs"]),
    "P99_abs":           float(glob["p99_abs"]),
    "parametri":         int(n_foglie),
    "parametri_nota":    "foglie totali; struttura dei tagli non contata",
    "tempo_train_s":     round(TEMPO_TRAIN_B4, 1),
    "flow_addestramento": int(len(y_all)),
    "alberi":            N_ALBERI,
    "learning_rate":     params_b4["learning_rate"],
    "num_leaves":        params_b4["num_leaves"],
    "min_data_in_leaf":  params_b4["min_data_in_leaf"],
    "min_data_per_group": params_b4["min_data_per_group"],
    "cat_smooth":        params_b4["cat_smooth"],
    "max_cat_threshold": 32,
    "mae_tune_selezione": 1.5944,
    "soglie_rmse_robusto": [float(LO), float(HI)],
}
with open(OUT_DIR / "B4_lightgbm.json", "w", encoding="utf-8") as f:
    json.dump(riepilogo_b4, f, ensure_ascii=False, indent=2)

# modello
MOD_DIR = PROJECT_ROOT / "results" / "modelli"
MOD_DIR.mkdir(parents=True, exist_ok=True)
b4.save_model(str(MOD_DIR / "B4_lightgbm.txt"))

print("salvati in", OUT_DIR, "e", MOD_DIR)
print(json.dumps(riepilogo_b4, ensure_ascii=False, indent=2))

salvati in /Users/giovannipavan/Desktop/EPFL/Corsi/Primo Semestre/ML/projects/flow-tokenization/results/metriche e /Users/giovannipavan/Desktop/EPFL/Corsi/Primo Semestre/ML/projects/flow-tokenization/results/modelli
{
  "modello": "B4 LightGBM (L1)",
  "MAE": 1.8332570069765324,
  "RMSE": 47.07572434124263,
  "RMSE_robusto": 4.94458663190812,
  "P90_abs": 4.563222661904012,
  "P99_abs": 22.640237498110384,
  "parametri": 25893,
  "parametri_nota": "foglie totali; struttura dei tagli non contata",
  "tempo_train_s": 31,
  "flow_addestramento": 4000667,
  "alberi": 411,
  "learning_rate": 0.1,
  "num_leaves": 63,
  "min_data_in_leaf": 50,
  "min_data_per_group": 50,
  "cat_smooth": 10,
  "max_cat_threshold": 32,
  "mae_tune_selezione": 1.5944,
  "soglie_rmse_robusto": [
    -184.23263,
    58.82300000000002
  ]
}


### 14. Verifica descrittiva: `application` e durata dei flow
Ipotesi: `application` è importante perché indica il tipo di lavoro (trasferimento di massa vs lettura per analisi), quindi indirettamente la durata e la moda di `delta_rtt`.
Per le 15 applicazioni più frequenti in validation: quota di flow > 1 min, durata mediana, `delta_rtt` mediano. Analisi solo descrittiva, nessuna scelta di modello.

In [62]:
tab_app = (val.assign(lungo=val["an_durata_s"] > 60)
              .groupby("id_application")
              .agg(n=("delta_rtt", "size"),
                   quota_lunghi=("lungo", "mean"),
                   durata_mediana_s=("an_durata_s", "median"),
                   delta_mediano=("delta_rtt", "median"))
              .sort_values("n", ascending=False)
              .head(15)
              .round(3))
tab_app

,n,quota_lunghi,durata_mediana_s,delta_mediano
id_application,,,,
4,4688359,0.967,539.340,5.089
5,1606253,0.199,2.346,0.008
6,1470876,0.423,14.605,5.047
7,763636,0.242,0.639,0.070
9,382382,0.498,57.096,0.361
11,232295,0.622,276.480,4.974
14,145134,0.756,171.476,3.548
12,142944,1.000,462.323,4.070
20,139974,0.984,585.996,4.648
